In [ ]:
%matplotlib widget
import numpy 
import pandas
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets
import os
import cryspy
from ipyfilechooser import FileChooser
from magic_tof import model_MAGiC as model_magic
import scipp 
import widget_code_cell
import matplotlib
import matplotlib.colors

import np_cryst_functions

In [84]:
def hkl_to_rgb(h,k,l):
    hkl = numpy.sqrt(numpy.square(h) + numpy.square(k) + numpy.square(l))
    hh_r = numpy.abs(h) / hkl
    hh_g = numpy.abs(k) / hkl
    hh_b = numpy.abs(l) / hkl
    return numpy.stack([hh_r, hh_g, hh_b], axis=1)

In [ ]:
def simulate_detector(intensities, gammas, nus, 
                      gamma_grid, nu_grid, sigma):
    """
    Simulate a diffraction pattern on a 2D detector using Gaussian peaks.

    Parameters
    ----------
    intensities : array-like
        Integrated intensities of reflections (I_i).
    gammas : array-like
        Gamma positions of reflections.
    nus : array-like
        Nu positions of reflections.
    gamma_grid : 2D ndarray
        Meshgrid of gamma coordinates on detector.
    nu_grid : 2D ndarray
        Meshgrid of nu coordinates on detector.
    sigma : float
        Gaussian peak width (constant).

    Returns
    -------
    pattern : 2D ndarray
        Simulated detector intensity.
    """

    pattern = numpy.zeros_like(gamma_grid, dtype=float)

    for I, g0, n0 in zip(intensities, gammas, nus):
        # Gaussian peak
        pattern += I * numpy.exp(
            -((gamma_grid - g0)**2 + (nu_grid - n0)**2) / (2 * sigma**2)
        )

    return pattern


def plot_pattern(pattern, gamma_grid, nu_grid):
    # Determine axis limits from the meshgrid
    pattern = numpy.clip(pattern, 1e-12, None)
    gamma_min, gamma_max = gamma_grid.min(), gamma_grid.max()
    nu_min, nu_max = nu_grid.min(), nu_grid.max()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(
        pattern,
        extent=(gamma_min, gamma_max, nu_min, nu_max),
        origin="lower",
        # cmap="inferno",
        aspect="auto",
        norm=matplotlib.colors.LogNorm(vmin=pattern.max()*1e-4, vmax=pattern.max()) 
    )
    fig.colorbar(im, ax=ax, label="Intensity")
    ax.set_xlabel("gamma")
    ax.set_ylabel("nu")
    ax.set_title("Simulated Diffraction Pattern")
    return fig



In [ ]:
def plot_scatters_with_labels(x, y, labels, color='red', size=4, n_visible=10, fig=None, color_text='red', marker='o'):
    if fig is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        ax = fig.axes[0]
    sc = ax.scatter(x, y, s=size, color=color,marker=marker)

    active_texts = []

    x = numpy.asarray(x)
    y = numpy.asarray(y)
    labels = numpy.asarray(labels)

    def update_labels(event=None):
        nonlocal active_texts

        # Remove old labels
        for t in active_texts:
            t.remove()
        active_texts = []

        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()

        mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        idx = numpy.where(mask)[0]

        # Only label if few points are visible
        if len(idx) <= n_visible:

            # --- GROUP POINTS BY (x, y) ---
            groups = {}
            for i in idx:
                key = (numpy.round(x[i],3), numpy.round(y[i],3))
                if key not in groups:
                    groups[key] = []
                groups[key].append(labels[i])

            # --- CREATE ONE LABEL PER GROUP ---
            for (xi, yi), labs in groups.items():
                # Split into chunks of 5 labels
                chunks = [labs[i:i+5] for i in range(0, len(labs), 5)]
                # Join each chunk with commas, then join chunks with newlines
                merged_label = "\n".join(", ".join(chunk) for chunk in chunks)                
                # merged_label = ", ".join(labs)
                t = ax.text(xi, yi, merged_label,
                            fontsize=8, color=color_text)
                active_texts.append(t)

        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("draw_event", update_labels)
    fig.canvas.mpl_connect("button_release_event", update_labels)
    fig.canvas.mpl_connect("scroll_event", update_labels)

    update_labels()
    return fig


In [ ]:
# ---------------------------
# Widgets
# ---------------------------
dir_path_data = "/Users/iuriikibalin/Desktop"

if not os.path.isdir(dir_path_data):
    dir_path_data = "/"



psc_nu_dropdown = ipywidgets.Dropdown(description="PSC_nu", options=[14,28,70,112, 154], value=14, disabled=False)
psc_opening_angle_floattext = ipywidgets.BoundedFloatText(description="Opening angle of PSC", min = 8.6, max = 105, step=0.1, value=105)
lambda_min_w = ipywidgets.FloatText(description="λ_min", value=0.5)


show_choppers_button = ipywidgets.Button(description="Choppers", button_style="info")
show_pulse_propagation_button = ipywidgets.Button(description="Pulse Propagation", button_style="info")
show_sample_spectra_button = ipywidgets.Button(description="Spectra on sample", button_style="info")
show_tof_resolution_button = ipywidgets.Button(description="Resolution", button_style="info")

sample_radiobutton = ipywidgets.RadioButtons(options=['By Unit Cell', 'By CIF'], orientation='horizontal', description="Sample input:")

fc = FileChooser(
    dir_path_data,
    title="Select CIF File (Optional)",
    select_default=False,
    layout={'display': 'none'},
)

cell_a_w = ipywidgets.FloatText(description="a", value=14.0,)
cell_b_w = ipywidgets.FloatText(description="b", value=14.0,)
cell_c_w = ipywidgets.FloatText(description="c", value=14.0,)

cell_alpha_w = ipywidgets.FloatText(description="α (deg)", value=90.0,)
cell_beta_w  = ipywidgets.FloatText(description="β (deg)", value=90.0,)
cell_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=90.0,)


ea_alpha_w = ipywidgets.FloatText(description="α (deg)", value=0.0)
ea_beta_w  = ipywidgets.FloatText(description="β (deg)", value=0.0)
ea_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=0.0)

omega_w = ipywidgets.FloatText(description="ω (deg)", value=0.0)
chi_w   = ipywidgets.FloatText(description="χ (deg)", value=0.0)
phi_w   = ipywidgets.FloatText(description="φ (deg)", value=0.0)


gamma_min_w = ipywidgets.FloatText(description="γ_min", value=10)
gamma_max_w = ipywidgets.FloatText(description="γ_max", value=70)
nu_min_w = ipywidgets.FloatText(description="ν_min", value=-35)
nu_max_w = ipywidgets.FloatText(description="ν_max", value=15)


cb_simulate = ipywidgets.Checkbox(value=False, description="Simulate detector image")


button = ipywidgets.Button(description="Generate peaks", button_style="success")

hkl_w = ipywidgets.Text(
    value='',
    placeholder='2 0 0',
    description='HKL:',
    disabled=False
)

button_hkl = ipywidgets.Button(description="Calc angles for HKL", button_style="success")

clean_button= ipywidgets.Button(description="Clean", button_style="danger")

out = ipywidgets.Output()

In [ ]:
def get_magic_tof_model_from_widgets_without_run(neutrons:int=10_000_000, pulses:int=1):
    psc_nu = psc_nu_dropdown.value
    psc_opening_angle = psc_opening_angle_floattext.value
    wavelength_band_min = lambda_min_w.value
    model = model_magic(
            psc_opening_angle=psc_opening_angle,
            psc_nu=psc_nu, 
            wavelength_band_min=wavelength_band_min,
            pulses=pulses,
            neutrons=neutrons,
            )
    return model

def get_magic_tof_model_from_widgets(neutrons:int=10_000_000, pulses:int=1):
    model = get_magic_tof_model_from_widgets_without_run(neutrons=neutrons, pulses=pulses)
    res = model.run()
    return res

def get_spectra_on_sample():
    model = get_magic_tof_model_from_widgets()
    data_flatten = model.detectors['Cave BM'].data.flatten(to='event2')
    data_reduced = data_flatten[~data_flatten.masks['blocked_by_others']]
    wavelength_min = data_reduced.coords["wavelength"].min().value
    wavelength_max = data_reduced.coords["wavelength"].max().value

    wavelength_point = int(numpy.round((wavelength_max-wavelength_min)/0.01, 0))+1
    bin_wavelength = scipp.linspace('wavelength', wavelength_min, wavelength_max, num=wavelength_point, unit='Å', endpoint=True)
    data_hist = data_reduced.hist(wavelength=bin_wavelength)    
    return data_hist, wavelength_min, wavelength_max


In [ ]:
# ---------------------------
# Callback

def parse_hkl_text(text):
    try:
        values = [float(v) for v in text.strip().split()[:3]]
        if len(values) != 3:
            raise ValueError("Expected three HKL values")
        return values
    except Exception:
        fallback = [float(v) for v in hkl_w.placeholder.strip().split()[:3]]
        return fallback

def format_matrix(matrix):
    return "\n".join(
        "".join(f"{v:9.5f}" for v in row)
        for row in matrix
    )

def deg2rad_widget_values(*widgets):
    return [numpy.deg2rad(w.value) for w in widgets]

def update_unit_cell_widgets_from_cif(file_path):
    rcif_obj = cryspy.file_to_globaln(file_path)
    crystal = [item for item in rcif_obj.items if isinstance(item, cryspy.Crystal)][0]
    ucp = crystal.get_dictionary()['unit_cell_parameters']
    cell_a_w.value = ucp[0]
    cell_b_w.value = ucp[1]
    cell_c_w.value = ucp[2]
    cell_alpha_w.value = numpy.rad2deg(ucp[3])
    cell_beta_w.value = numpy.rad2deg(ucp[4])
    cell_gamma_w.value = numpy.rad2deg(ucp[5])

def show_choppers_clicked(_):
    with out:
        out.clear_output()
        model = get_magic_tof_model_from_widgets_without_run()
        for chopper_name, chopper in model.choppers.items():
            display(chopper.to_diskchopper())

def show_pulse_propagation_clicked(_):
    with out:
        out.clear_output()
        model = get_magic_tof_model_from_widgets(pulses=2)
        fig = model.plot(blocked_rays=5000)
        display(fig.fig)

def show_sample_spectra_clicked(_):
    with out:
        out.clear_output()
        data_hist, wavelength_min, wavelength_max = get_spectra_on_sample()
        fig = data_hist.plot()
        display(fig)

def show_tof_resolution_clicked(_):
    with out:
        out.clear_output()
        model = get_magic_tof_model_from_widgets()
        da_events = model.detectors['Cave BM'].data.flatten(to='event')
        da_events = da_events[~da_events.masks['blocked_by_others']]
        fig = da_events.hist(wavelength=500, toa=500).plot(norm='log', grid=True)
        display(fig)

def compute_angles(_):
    with out:
        out.clear_output()
        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value

        cell_alpha, cell_beta, cell_gamma = deg2rad_widget_values(cell_alpha_w, cell_beta_w, cell_gamma_w)
        ea_alpha, ea_beta, ea_gamma = deg2rad_widget_values(ea_alpha_w, ea_beta_w, ea_gamma_w)
        sample_omega, sample_chi, sample_phi = deg2rad_widget_values(omega_w, chi_w, phi_w)

        # --- compute matrices ---
        m_b = np_cryst_functions.calc_b_matrix(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma)
        m_u = np_cryst_functions.calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b

        m_r = np_cryst_functions.calc_sample_rotation(sample_omega, sample_chi, sample_phi)
        h, k, l = parse_hkl_text(hkl_w.value)

        gamma, nu, wavelength = np_cryst_functions.calc_gamma_nu_wavelength_for_hkl(h, k, l, m_ub, m_r)
        print(f"# Peak ({h:.0f} {k:.0f} {l:.0f})")
        print("\nUB matrix:")
        print(format_matrix(m_ub))
        print("\nRotation matrix:")
        print(format_matrix(m_r))
        print(f"\nGamma: {gamma.squeeze():.2f} deg")
        print(f"Nu:    {nu.squeeze():.2f} deg")
        print(f"Wavelength: {wavelength.squeeze():.5f} Å")

def on_file_selected(_):
    file_path = fc.selected

    with out:
        out.clear_output()
        if file_path is None or not os.path.isfile(file_path):
            return
        try:
            update_unit_cell_widgets_from_cif(file_path)
        except Exception as exc:
            print(f"Failed to load CIF: {exc}")

    return

def compute(_):
    with out:
        global data, da_peaks
        out.clear_output()

        file_path = fc.selected
        flag_crystal = False
        if file_path is None or not os.path.isfile(file_path):
            # --- read inputs ---
            pass
        else:
            try:
                rcif_obj = cryspy.file_to_globaln(file_path)
                crystal = [hh for hh in rcif_obj.items if isinstance(hh, cryspy.Crystal)][0]
                flag_crystal = True
            except Exception:
                flag_crystal = False

        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value
        cell_alpha = numpy.deg2rad(cell_alpha_w.value)
        cell_beta = numpy.deg2rad(cell_beta_w.value)
        cell_gamma = numpy.deg2rad(cell_gamma_w.value)
        ea_alpha = numpy.deg2rad(ea_alpha_w.value)
        ea_beta = numpy.deg2rad(ea_beta_w.value)
        ea_gamma = numpy.deg2rad(ea_gamma_w.value)

        sample_omega = numpy.deg2rad(omega_w.value)
        sample_chi = numpy.deg2rad(chi_w.value)
        sample_phi = numpy.deg2rad(phi_w.value)

        da_spectra_hist, lambda_min, lambda_max = get_spectra_on_sample()

        # --- compute limits ---
        hmax = int(numpy.round(2 * cell_a / lambda_min, 0))
        kmax = int(numpy.round(2 * cell_b / lambda_min, 0))
        lmax = int(numpy.round(2 * cell_c / lambda_min, 0))

        # --- compute matrices ---
        m_b = np_cryst_functions.calc_b_matrix(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma)
        m_u = np_cryst_functions.calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b
        m_r = np_cryst_functions.calc_sample_rotation(sample_omega, sample_chi, sample_phi)

        # --- generate peak data ---
        data = np_cryst_functions.generate_peak_data(
            UB=m_ub,
            R=m_r,
            hmax=hmax, kmax=kmax, lmax=lmax,
            lambda_min=lambda_min,
            lambda_max=lambda_max
        )

        mask = numpy.logical_and(data[:,3] >= gamma_min_w.value, data[:,3] <= gamma_max_w.value)
        data = data[mask]
        mask = numpy.logical_and(data[:,4] >= nu_min_w.value, data[:,4] <= nu_max_w.value)
        data = data[mask]
        
        cos_tth = numpy.cos(numpy.radians(data[:,3]))*numpy.cos(numpy.radians(data[:,4]))
        if flag_crystal:
            fn = crystal.calc_f_nucl(numpy.stack([data[:,0], data[:,1], data[:,2]], axis=0))
            fn_sq = numpy.square(numpy.abs(fn))
            np_spectra = numpy.interp(data[:,5], scipp.midpoints(da_spectra_hist.coords['wavelength']).values, da_spectra_hist.data.values, left=0., right=0.)

            th = 0.5*numpy.acos(cos_tth)
            d_hkl = data[:,5]/(2*numpy.sin(th))
            lf = numpy.pow(d_hkl,4)/numpy.sin(th)
            iint = lf*fn_sq * np_spectra
            data = numpy.column_stack((data, iint))

        size = 4
        if data.shape[1]>=7:
            mask = data[:,6] >= 0.00001*data[:,6].max()
            data = data[mask]
            size = 5. * size * data[:,6]/data[:,6].max()

        # --- output ---
        if data.shape[1]>=7:
            if cb_simulate.value:
                gamma = numpy.linspace(gamma_min_w.value, gamma_max_w.value, num=int((gamma_max_w.value-gamma_min_w.value)/0.2),endpoint=True)
                nu= numpy.linspace(nu_min_w.value, nu_max_w.value, num=int((nu_max_w.value-nu_min_w.value)/0.2),endpoint=True)
                gamma_grid, nu_grid = numpy.meshgrid(gamma, nu)
                pattern = simulate_detector(data[:,6], data[:,3], data[:,4], gamma_grid,nu_grid, sigma=0.25)
                fig = plot_pattern(pattern, gamma_grid,nu_grid)
                size = 0.2
                color = 'white'
                color_text = 'white'
            else:
                fig = None
                color = hkl_to_rgb(data[:,0], data[:,1], data[:,2])
                color_text = 'black'

            labels = [f"({h:.0f} {k:.0f} {l:.0f})" for h,k,l in zip(data[:,0], data[:,1], data[:,2])]
            fig = plot_scatters_with_labels(data[:,3], data[:,4], labels, color=color, size=size, n_visible=20, fig=fig, color_text=color_text)
            df = pandas.DataFrame(data, columns=['H', 'K', 'L', 'gamma', 'nu', 'wavelength', 'iint'])

        else:
            labels = [f"({h:.0f} {k:.0f} {l:.0f})" for h,k,l in zip(data[:,0], data[:,1], data[:,2])]
            np_rgb = hkl_to_rgb(data[:,0], data[:,1], data[:,2])
            fig = plot_scatters_with_labels(data[:,3], data[:,4], labels, color=np_rgb, size=size, n_visible=20)
            df = pandas.DataFrame(data, columns=['H', 'K', 'L', 'gamma', 'nu', 'wavelength'])
        fig.axes[0].set_xlim((gamma_min_w.value, gamma_max_w.value))
        fig.axes[0].set_ylim((nu_min_w.value, nu_max_w.value))

        display(fig.canvas)
        da_peaks = scipp.compat.pandas_compat.from_pandas(df)
        display(scipp.table(da_peaks))

def on_change_sample(change):
    if change['name'] == 'value':
        if change['new'] == 'By CIF':
            cell_a_w.layout.display = 'none'
            cell_b_w.layout.display = 'none'
            cell_c_w.layout.display = 'none'
            cell_alpha_w.layout.display = 'none'
            cell_beta_w.layout.display = 'none'
            cell_gamma_w.layout.display = 'none'
            fc.layout.display = 'block'
        else:
            fc.layout.display = 'none'
            cell_a_w.layout.display = 'block'
            cell_b_w.layout.display = 'block'
            cell_c_w.layout.display = 'block'
            cell_alpha_w.layout.display = 'block'
            cell_beta_w.layout.display = 'block'
            cell_gamma_w.layout.display = 'block'

def clean_clicked(_):
    with out:
        out.clear_output()

In [ ]:
sample_radiobutton.observe(on_change_sample)
fc.register_callback(on_file_selected)

show_choppers_button.on_click(show_choppers_clicked)
show_pulse_propagation_button.on_click(show_pulse_propagation_clicked)
show_sample_spectra_button.on_click(show_sample_spectra_clicked)
show_tof_resolution_button.on_click(show_tof_resolution_clicked)

button.on_click(compute)
button_hkl.on_click(compute_angles)
clean_button.on_click(clean_clicked)


In [ ]:
accordion = ipywidgets.Accordion(
    children=[
        ipywidgets.VBox([
            ipywidgets.HBox([psc_nu_dropdown, psc_opening_angle_floattext, lambda_min_w]),
            ipywidgets.HBox([show_choppers_button, show_pulse_propagation_button, show_sample_spectra_button,show_tof_resolution_button, ]),
        ]), 
        ipywidgets.VBox([
            sample_radiobutton,
            fc, 
            ipywidgets.HBox([cell_a_w, cell_b_w, cell_c_w]),
            ipywidgets.HBox([cell_alpha_w, cell_beta_w, cell_gamma_w]),
            ipywidgets.HTML("<h3>Sample Orientation (Euler angles)</h3>"),
            ipywidgets.HBox([ea_alpha_w, ea_beta_w, ea_gamma_w]),
            ipywidgets.HTML("<h3>Goniometer Rotation</h3>"),
            ipywidgets.HBox([omega_w, chi_w, phi_w]),
        ]), 
        ipywidgets.VBox([
            ipywidgets.HBox([gamma_min_w, gamma_max_w,]),
            ipywidgets.HBox([nu_min_w, nu_max_w,]),
        ])
        ], 
        titles=('Chopper Settings', 'Sample', 'Detector'))


ui = ipywidgets.VBox([
    ipywidgets.HTML("<h1>MAGiC Planner</h1>"),
    accordion,
    ipywidgets.HTML("<h2></h2>"),
    ipywidgets.HBox([button, cb_simulate]),
    ipywidgets.HBox([button_hkl, hkl_w]),
    clean_button,
    ipywidgets.HTML("<h2></h2>"),
    out,
])

display(ui)

In [ ]:
widget_code_cell.make_code_cell(
    width="80%", 
    placeholder="Write Python code here ...\n\nThe follwoing global variables are available:\n - da_peaks (stores peaks)\n - data",
    )